# S4 · AndinaLog 03B · Notebook 1 · Diagnóstico de seguimiento de inventario

Este notebook trabaja **solo** con `andinalog_inventory_tracking.csv`. Lee la capa Bronze desde `datasets/AndinaLog_03B_Bronce/`, detecta problemas y conserva los doce campos originales. No convierte formatos, no imputa ni corrige datos. El tratamiento de los casos recuperables corresponde al notebook 2, después de aprobar sus reglas.

Cada ejecución reemplaza cuatro archivos en `S4/andinalog_inventory_tracking/notebook1/salidas/`:

1. `andinalog_inventory_tracking_diagnosticado.csv`: todas las filas, los campos originales y solo `fila_bronze`, `en_cuarentena` y `columnas_con_problemas`.
2. `andinalog_inventory_tracking_problemas.csv`: una fila por problema, con columna, código estable y evidencia.
3. `andinalog_inventory_tracking_cuarentena.csv`: extracto informativo de las filas marcadas.
4. `andinalog_inventory_tracking_reporte_calidad.csv`: conteos y huella SHA-256 del CSV de origen.

## 1 · Configuración y origen

En local, ejecuta el notebook desde cualquier carpeta dentro del proyecto. En Colab, monta Drive, selecciona `ENTORNO = "drive"` y ajusta `RUTA_PROYECTO_DRIVE` a la carpeta que contiene `datasets/` y `S4/`. Las salidas van a `S4/andinalog_inventory_tracking/notebook1/salidas/` en ese mismo entorno.

In [2]:
from pathlib import Path
import hashlib
import os
import re
import tempfile
import pandas as pd

ENTORNO = "local"  # "local" o "drive"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"  # ajustar si la carpeta real es otra
CARPETA_DATASETS = "AndinaLog_03B_Bronce"
NOMBRE_CSV = "andinalog_inventory_tracking.csv"
VERSION_DIAGNOSTICO = "GIAD-M3-S4-INV-diagnostico-v1"

COLUMNAS_ORIGINALES = [
    "movimiento_id", "lote_id", "producto_id", "centro_distribucion",
    "fecha_ingreso", "fecha_salida", "fecha_vencimiento",
    "cantidad_ingreso", "cantidad_salida", "cantidad_merma",
    "dias_en_almacen", "costo_unitario_bob"
]

def encontrar_raiz_local():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "datasets" / CARPETA_DATASETS).is_dir() and (carpeta / "S4").is_dir():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz del proyecto; ejecuta dentro de practicasNotebookColab.")

def configurar_rutas(entorno, ruta_drive):
    if entorno == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(ruta_drive)
    elif entorno == "local":
        raiz = encontrar_raiz_local()
    else:
        raise ValueError("ENTORNO debe ser 'local' o 'drive'")
    bronze = raiz / "datasets" / CARPETA_DATASETS / NOMBRE_CSV
    salidas = raiz / "S4" / "andinalog_inventory_tracking" / "notebook1" / "salidas"
    if not bronze.is_file():
        raise FileNotFoundError(f"No se encontró el CSV Bronze: {bronze}")
    return bronze, salidas

RUTA_BRONZE, DIRECTORIO_SALIDAS = configurar_rutas(ENTORNO, RUTA_PROYECTO_DRIVE)
print("Bronze:", RUTA_BRONZE)
print("Salidas:", DIRECTORIO_SALIDAS)

Bronze: c:\Users\remrodri\Github\practicasNotebookColab\datasets\AndinaLog_03B_Bronce\andinalog_inventory_tracking.csv
Salidas: c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_inventory_tracking\notebook1\salidas


## 2 · Carga y contrato

La lectura mantiene todas las columnas como texto y los vacíos como cadenas vacías. Las conversiones numéricas y de fecha usadas para comprobar errores son temporales: no se exportan como valores transformados.

In [3]:
def cargar_bronze(ruta):
    huella = hashlib.sha256(ruta.read_bytes()).hexdigest()
    df = pd.read_csv(ruta, dtype="string", encoding="utf-8-sig", keep_default_na=False)
    return df, huella

def validar_esquema(df):
    if list(df.columns) != COLUMNAS_ORIGINALES:
        faltantes = sorted(set(COLUMNAS_ORIGINALES) - set(df.columns))
        extras = sorted(set(df.columns) - set(COLUMNAS_ORIGINALES))
        raise ValueError(f"Esquema inesperado. Faltantes: {faltantes}; extras: {extras}; orden: {list(df.columns)}")
    if not df.columns.is_unique:
        raise ValueError("Hay nombres de columnas duplicados")
    return df

df_bronze, HASH_BRONZE = cargar_bronze(RUTA_BRONZE)
validar_esquema(df_bronze)
print(f"Bronze: {len(df_bronze):,} filas × {len(df_bronze.columns)} columnas")
print("SHA-256:", HASH_BRONZE)
display(df_bronze.head())

Bronze: 6,040 filas × 12 columnas
SHA-256: 6b10604538376dda79b3ebe4cc77015321c31ae17ac092e29edf3c66a5cf2c0c


,movimiento_id,lote_id,producto_id,centro_distribucion,fecha_ingreso,fecha_salida,fecha_vencimiento,cantidad_ingreso,cantidad_salida,cantidad_merma,dias_en_almacen,costo_unitario_bob
0,MOV-000001,LOT-2026-00001,PROD-017,Tarija,2026-06-02,2026-06-08,2026-06-20,681,660,18,6,19.81
1,MOV-000002,LOT-2026-00002,PROD-059,Santa Cruz,2026-07-16,2026-07-26,2026-08-03,287,274,8,10,13.44
2,MOV-000003,LOT-2026-00003,PROD-022,Oruro,2026-05-11,2026-05-18,2026-11-07,177,173,2,7,67.52
3,MOV-000004,LOT-2026-00004,PROD-038,Santa Cruz,2026-07-21,2026-08-25,2027-07-21,509,502,3,35,13.49
4,MOV-000005,LOT-2026-00005,PROD-024,La Paz,2026-06-07,2026-06-25,2027-06-07,775,763,8,18,23.48


## 3 · Catálogo y reglas de diagnóstico

Los códigos son estables para que el notebook 2 pueda reconocer el problema exacto. `columna_afectada` identifica la columna o clave primaria. Las reglas auditan nulos, formatos heterogéneos de fechas, fechas no calendario, duplicados de clave y rangos de inventario. Detectan; no deciden todavía cómo corregir.

In [4]:
CATALOGO_PROBLEMAS = pd.DataFrame([
    ("movimiento_id", "FALTANTE", "Identificador vacío"),
    ("movimiento_id", "DUPLICADO", "Identificador de movimiento repetido; se marca la aparición posterior"),
    ("lote_id", "FALTANTE", "Identificador de lote vacío"),
    ("producto_id", "FALTANTE", "Identificador de producto vacío"),
    ("centro_distribucion", "FALTANTE", "Centro de distribución no informado"),
    ("fecha_ingreso", "FALTANTE", "Fecha de ingreso vacía"),
    ("fecha_ingreso", "FECHA_INVALIDA", "Formato no ISO o fecha inexistente"),
    ("fecha_salida", "FECHA_INVALIDA", "Formato no ISO o fecha inexistente (en registros con salida)"),
    ("fecha_vencimiento", "FALTANTE", "Fecha de vencimiento vacía"),
    ("fecha_vencimiento", "FECHA_INVALIDA", "Formato no ISO o fecha inexistente"),
    ("cantidad_ingreso", "FALTANTE", "Campo vacío"),
    ("cantidad_ingreso", "NO_NUMERICA", "Valor no convertible a número entero"),
    ("cantidad_ingreso", "FUERA_RANGO", "Cantidad menor o igual a cero"),
    ("cantidad_salida", "FALTANTE", "Campo vacío"),
    ("cantidad_salida", "NO_NUMERICA", "Valor no convertible a número entero"),
    ("cantidad_salida", "FUERA_RANGO", "Cantidad negativa"),
    ("cantidad_merma", "FALTANTE", "Campo vacío"),
    ("cantidad_merma", "NO_NUMERICA", "Valor no convertible a número"),
    ("cantidad_merma", "FUERA_RANGO", "Merma física negativa o fuera de rango"),
    ("dias_en_almacen", "FALTANTE", "Campo vacío"),
    ("dias_en_almacen", "NO_NUMERICA", "Valor no convertible a número entero"),
    ("dias_en_almacen", "FUERA_RANGO", "Días negativos"),
    ("costo_unitario_bob", "FALTANTE", "Campo vacío"),
    ("costo_unitario_bob", "NO_NUMERICA", "Valor no convertible a número"),
    ("costo_unitario_bob", "FUERA_RANGO", "Costo unitario menor o igual a cero"),
], columns=["columna_afectada", "codigo_error", "criterio"])
display(CATALOGO_PROBLEMAS)

def registrar_problema(df, mascara, columna, codigo, evidencia=None):
    mascara = mascara.fillna(False).astype(bool)
    filas = df.loc[mascara, ["fila_bronze"]].copy()
    filas["columna_afectada"] = columna
    filas["codigo_error"] = codigo
    if evidencia is None:
        evidencia = df[columna] if columna in df.columns else pd.Series("", index=df.index, dtype="string")
    filas["valor_original"] = evidencia.loc[mascara].astype("string").to_numpy()
    return filas

def texto(df, columna):
    return df[columna].astype("string").str.strip()

def detectar_identificadores(df):
    return [registrar_problema(df, texto(df, col).eq(""), col, "FALTANTE")
            for col in ["movimiento_id", "lote_id", "producto_id", "centro_distribucion"]]

def detectar_fechas_y_duplicados(df):
    hallazgos = []
    # Clave duplicada por movimiento_id
    clave = texto(df, "movimiento_id")
    duplicada = clave.duplicated(keep="first")
    hallazgos.append(registrar_problema(df, duplicada, "movimiento_id", "DUPLICADO", clave))

    # fecha_ingreso
    fi = texto(df, "fecha_ingreso")
    fi_fmt = fi.str.fullmatch(r"\d{4}-\d{2}-\d{2}").fillna(False)
    fi_dt = pd.to_datetime(fi, format="%Y-%m-%d", errors="coerce")
    hallazgos.append(registrar_problema(df, fi.eq(""), "fecha_ingreso", "FALTANTE"))
    hallazgos.append(registrar_problema(df, fi.ne("") & (~fi_fmt | fi_dt.isna()), "fecha_ingreso", "FECHA_INVALIDA"))

    # fecha_salida (puede ser vacía si el lote sigue en almacén; si tiene valor debe ser válida)
    fs = texto(df, "fecha_salida")
    fs_fmt = fs.str.fullmatch(r"\d{4}-\d{2}-\d{2}").fillna(False)
    fs_dt = pd.to_datetime(fs, format="%Y-%m-%d", errors="coerce")
    hallazgos.append(registrar_problema(df, fs.ne("") & (~fs_fmt | fs_dt.isna()), "fecha_salida", "FECHA_INVALIDA"))

    # fecha_vencimiento
    fv = texto(df, "fecha_vencimiento")
    fv_fmt = fv.str.fullmatch(r"\d{4}-\d{2}-\d{2}").fillna(False)
    fv_dt = pd.to_datetime(fv, format="%Y-%m-%d", errors="coerce")
    hallazgos.append(registrar_problema(df, fv.eq(""), "fecha_vencimiento", "FALTANTE"))
    hallazgos.append(registrar_problema(df, fv.ne("") & (~fv_fmt | fv_dt.isna()), "fecha_vencimiento", "FECHA_INVALIDA"))

    return hallazgos

def detectar_medicion(df, columna, rango=None):
    valor = texto(df, columna)
    numero = pd.to_numeric(valor, errors="coerce")
    hallazgos = [
        registrar_problema(df, valor.eq(""), columna, "FALTANTE"),
        registrar_problema(df, valor.ne("") & numero.isna(), columna, "NO_NUMERICA"),
    ]
    if rango is not None:
        minimo, maximo = rango
        hallazgos.append(registrar_problema(df, numero.notna() & ~numero.between(minimo, maximo), columna, "FUERA_RANGO"))
    return hallazgos

def diagnosticar(df_bronze):
    principal = df_bronze.copy(deep=True)
    principal.insert(0, "fila_bronze", range(1, len(principal) + 1))
    hallazgos = (
        detectar_identificadores(principal)
        + detectar_fechas_y_duplicados(principal)
        + detectar_medicion(principal, "cantidad_ingreso", rango=(1, 100000))
        + detectar_medicion(principal, "cantidad_salida", rango=(0, 100000))
        + detectar_medicion(principal, "cantidad_merma", rango=(0, 100000))
        + detectar_medicion(principal, "dias_en_almacen", rango=(0, 1000))
        + detectar_medicion(principal, "costo_unitario_bob", rango=(0.01, 100000.0))
    )
    problemas = pd.concat(hallazgos, ignore_index=True)
    problemas = problemas.sort_values(["fila_bronze", "columna_afectada", "codigo_error"], kind="stable").reset_index(drop=True)
    problemas["version_diagnostico"] = VERSION_DIAGNOSTICO
    columnas_por_fila = problemas.groupby("fila_bronze")["columna_afectada"].agg(
        lambda valores: "|".join(dict.fromkeys(valores))
    )
    principal["columnas_con_problemas"] = principal["fila_bronze"].map(columnas_por_fila).fillna("")
    principal["en_cuarentena"] = principal["columnas_con_problemas"].ne("")
    return principal, problemas

df_diagnosticado, df_problemas = diagnosticar(df_bronze)
df_cuarentena = df_diagnosticado.loc[df_diagnosticado["en_cuarentena"]].copy()
print(f"Principal: {len(df_diagnosticado):,}; problemas: {len(df_problemas):,}; filas en cuarentena: {len(df_cuarentena):,}")
display(df_problemas.groupby(["columna_afectada", "codigo_error"]).size().rename("filas").reset_index())

,columna_afectada,codigo_error,criterio
0,movimiento_id,FALTANTE,Identificador vacío
1,movimiento_id,DUPLICADO,Identificador de movimiento repetido; se marca...
2,lote_id,FALTANTE,Identificador de lote vacío
3,producto_id,FALTANTE,Identificador de producto vacío
4,centro_distribucion,FALTANTE,Centro de distribución no informado
5,fecha_ingreso,FALTANTE,Fecha de ingreso vacía
6,fecha_ingreso,FECHA_INVALIDA,Formato no ISO o fecha inexistente
7,fecha_salida,FECHA_INVALIDA,Formato no ISO o fecha inexistente (en registr...
8,fecha_vencimiento,FALTANTE,Fecha de vencimiento vacía
9,fecha_vencimiento,FECHA_INVALIDA,Formato no ISO o fecha inexistente


Principal: 6,040; problemas: 104; filas en cuarentena: 104


,columna_afectada,codigo_error,filas
0,cantidad_ingreso,NO_NUMERICA,10
1,cantidad_merma,FUERA_RANGO,6
2,fecha_ingreso,FECHA_INVALIDA,25
3,fecha_salida,FECHA_INVALIDA,5
4,fecha_vencimiento,FALTANTE,18
5,movimiento_id,DUPLICADO,40


## 4 · Reporte y comprobaciones antes de exportar

El reporte registra la huella SHA-256 para reconocer la versión exacta del CSV de origen. Los conteos de problemas pueden superar el número de filas en cuarentena porque una fila puede tener varios hallazgos simultáneos.

In [5]:
def construir_reporte(df_bronze, principal, problemas, ruta, huella):
    conteos = problemas.groupby(["columna_afectada", "codigo_error"]).size()
    datos = [
        ("archivo_bronze", ruta.name),
        ("sha256_bronze", huella),
        ("version_diagnostico", VERSION_DIAGNOSTICO),
        ("filas_bronze", len(df_bronze)),
        ("filas_diagnosticadas", len(principal)),
        ("filas_en_cuarentena", int(principal["en_cuarentena"].sum())),
        ("filas_sin_cuarentena", int((~principal["en_cuarentena"]).sum())),
        ("problemas_detectados", len(problemas)),
    ]
    datos += [(f"{col}:{codigo}", int(total)) for (col, codigo), total in conteos.items()]
    return pd.DataFrame(datos, columns=["metrica", "valor"])

def validar_resultados(df_bronze, principal, problemas, cuarentena, reporte):
    assert list(principal.columns) == ["fila_bronze", *COLUMNAS_ORIGINALES, "columnas_con_problemas", "en_cuarentena"]
    pd.testing.assert_frame_equal(principal[COLUMNAS_ORIGINALES], df_bronze[COLUMNAS_ORIGINALES])
    assert len(principal) == len(df_bronze)
    assert principal["fila_bronze"].is_unique
    assert len(cuarentena) == int(principal["en_cuarentena"].sum())
    assert problemas["fila_bronze"].isin(principal["fila_bronze"]).all()
    assert problemas[["columna_afectada", "codigo_error"]].apply(tuple, axis=1).isin(
        CATALOGO_PROBLEMAS[["columna_afectada", "codigo_error"]].apply(tuple, axis=1)
    ).all()
    assert set(problemas["fila_bronze"]) == set(cuarentena["fila_bronze"])
    assert len(reporte) >= 8

reporte_calidad = construir_reporte(df_bronze, df_diagnosticado, df_problemas, RUTA_BRONZE, HASH_BRONZE)
validar_resultados(df_bronze, df_diagnosticado, df_problemas, df_cuarentena, reporte_calidad)
display(reporte_calidad)
print("Comprobaciones previas a la exportación: correctas")


,metrica,valor
0,archivo_bronze,andinalog_inventory_tracking.csv
1,sha256_bronze,6b10604538376dda79b3ebe4cc77015321c31ae17ac092...
2,version_diagnostico,GIAD-M3-S4-INV-diagnostico-v1
3,filas_bronze,6040
4,filas_diagnosticadas,6040
5,filas_en_cuarentena,104
6,filas_sin_cuarentena,5936
7,problemas_detectados,104
8,cantidad_ingreso:NO_NUMERICA,10
9,cantidad_merma:FUERA_RANGO,6


Comprobaciones previas a la exportación: correctas


## 5 · Exportación reproducible

Los cuatro CSV se escriben primero como archivos temporales en `S4/andinalog_inventory_tracking/notebook1/salidas/` y se reemplazan con el mismo nombre al final. El CSV Bronze nunca se sobrescribe. Si se vuelve a ejecutar con la misma fuente y reglas, las salidas se actualizan en lugar de acumular versiones antiguas.

In [6]:
def exportar_salidas(directorio, tablas, ruta_bronze, huella_inicial):
    if hashlib.sha256(ruta_bronze.read_bytes()).hexdigest() != huella_inicial:
        raise RuntimeError("El CSV Bronze cambió durante la ejecución; no se exportarán resultados")
    directorio.mkdir(parents=True, exist_ok=True)
    temporales = {}
    try:
        for nombre, tabla in tablas.items():
            destino = directorio / nombre
            with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", prefix=".tmp_inv_", dir=directorio,
                                             encoding="utf-8-sig", newline="", delete=False) as tmp:
                tabla.to_csv(tmp, index=False)
                temporales[destino] = Path(tmp.name)
        for destino, temporal in temporales.items():
            os.replace(temporal, destino)
    finally:
        for temporal in temporales.values():
            temporal.unlink(missing_ok=True)
    return list(temporales)

tablas_salida = {
    "andinalog_inventory_tracking_diagnosticado.csv": df_diagnosticado,
    "andinalog_inventory_tracking_problemas.csv": df_problemas,
    "andinalog_inventory_tracking_cuarentena.csv": df_cuarentena,
    "andinalog_inventory_tracking_reporte_calidad.csv": reporte_calidad,
}
rutas_creadas = exportar_salidas(DIRECTORIO_SALIDAS, tablas_salida, RUTA_BRONZE, HASH_BRONZE)
for ruta in rutas_creadas:
    print(ruta)
print("Bronze intacta; salidas anteriores reemplazadas")

c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_inventory_tracking\notebook1\salidas\andinalog_inventory_tracking_diagnosticado.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_inventory_tracking\notebook1\salidas\andinalog_inventory_tracking_problemas.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_inventory_tracking\notebook1\salidas\andinalog_inventory_tracking_cuarentena.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_inventory_tracking\notebook1\salidas\andinalog_inventory_tracking_reporte_calidad.csv
Bronze intacta; salidas anteriores reemplazadas


## Siguiente etapa

El notebook 2 leerá el archivo diagnosticado y el detalle de problemas. El informe de S4 justifica tratamientos posibles, pero ninguna regla de curación está aprobada automáticamente por este diagnóstico. Una fila saldrá de cuarentena solo cuando todos sus problemas hayan sido resueltos y validados.
